In [14]:
import xgboost as xgb

In [15]:
import os
import json
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Tuple

import numpy as np
import pandas as pd
import wfdb
import matplotlib.pyplot as plt

from scipy.signal import welch
from scipy.stats import kurtosis, skew

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    average_precision_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

try:
    from xgboost import XGBClassifier
except Exception as e:
    raise ImportError(
        "xgboost belum ke-install. Install dulu: pip install xgboost"
    ) from e


# =========================================================
# CONFIG
# =========================================================
@dataclass
class Config:
    metadata_path: str = r"/Users/angga0704/VSCode/Lomba/IDSC/Misc/metadata.csv"
    files_dir: str = r"/Users/angga0704/VSCode/Lomba/IDSC/Misc/files"
    output_dir: str = "outputs_feature_importance"

    random_state: int = 42
    test_size: float = 0.20

    # pilih mode:
    # "all_12_leads"  -> buat lihat importance semua lead
    # "v1_v2_v3_only" -> buat importance khusus 3 lead
    mode: str = "all_12_leads"

    include_wavelet_features: bool = False

    # top feature plot
    top_n_features_plot: int = 20

    # permutation importance
    permutation_n_repeats: int = 20
    permutation_scoring: str = "roc_auc"


CFG = Config()


# =========================================================
# FILE UTILS
# =========================================================
def ensure_output_dirs(cfg: Config) -> Dict[str, Path]:
    base = Path(cfg.output_dir)
    paths = {
        "base": base,
        "plots": base / "plots",
        "tables": base / "tables",
    }
    for p in paths.values():
        p.mkdir(parents=True, exist_ok=True)
    return paths


def save_json(obj: Any, path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


# =========================================================
# DATA LOADING
# =========================================================
def load_metadata(cfg: Config) -> pd.DataFrame:
    df = pd.read_csv(cfg.metadata_path)

    required = {"patient_id", "brugada"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = df[df["brugada"].isin([0, 1])].copy()
    df["label"] = df["brugada"].astype(int)
    df = df.drop_duplicates(subset=["patient_id"]).reset_index(drop=True)

    return df


def load_ecg_record(patient_id: Any, files_dir: str) -> Tuple[np.ndarray, List[str], float]:
    pid = str(patient_id)
    record_path = os.path.join(files_dir, pid, pid)
    record = wfdb.rdrecord(record_path)
    return record.p_signal.astype(np.float32), list(record.sig_name), float(record.fs)


def lead_indices_from_names(lead_names: List[str], target_leads: List[str]) -> List[int]:
    mapping = {name: idx for idx, name in enumerate(lead_names)}
    return [mapping[name] for name in target_leads if name in mapping]


# =========================================================
# FEATURE EXTRACTION
# =========================================================
def safe_entropy(values: np.ndarray, bins: int = 32) -> float:
    hist, _ = np.histogram(values, bins=bins, density=True)
    hist = hist[hist > 0]
    if len(hist) == 0:
        return 0.0
    return float(-np.sum(hist * np.log(hist + 1e-12)))


def base_stats_1d(x: np.ndarray) -> List[float]:
    x = np.asarray(x)
    dx = np.diff(x)
    zero_cross = float(np.mean(np.diff(np.signbit(x)).astype(int))) if len(x) > 1 else 0.0

    return [
        float(np.mean(x)),
        float(np.std(x)),
        float(np.min(x)),
        float(np.max(x)),
        float(np.max(x) - np.min(x)),
        float(np.median(x)),
        float(np.percentile(x, 5)),
        float(np.percentile(x, 25)),
        float(np.percentile(x, 75)),
        float(np.percentile(x, 95)),
        float(np.mean(np.abs(x))),
        float(np.sqrt(np.mean(x ** 2))),
        float(np.sum(x ** 2)),
        float(skew(x)),
        float(kurtosis(x)),
        float(np.mean(dx)) if len(dx) else 0.0,
        float(np.std(dx)) if len(dx) else 0.0,
        float(np.max(np.abs(dx))) if len(dx) else 0.0,
        zero_cross,
        float(safe_entropy(x)),
    ]


def spectral_features_1d(x: np.ndarray, fs: float) -> List[float]:
    freqs, psd = welch(x, fs=fs, nperseg=min(len(x), 256))
    if len(freqs) == 0 or len(psd) == 0:
        return [0.0] * 6

    psd = np.maximum(psd, 1e-12)
    total_power = float(np.trapz(psd, freqs))

    def bandpower(low: float, high: float) -> float:
        mask = (freqs >= low) & (freqs < high)
        if not np.any(mask):
            return 0.0
        return float(np.trapz(psd[mask], freqs[mask]))

    power_0_5 = bandpower(0.0, 5.0)
    power_5_15 = bandpower(5.0, 15.0)
    power_15_40 = bandpower(15.0, 40.0)
    peak_freq = float(freqs[np.argmax(psd)])
    p = psd / psd.sum()
    spec_entropy = float(-np.sum(p * np.log(p + 1e-12)))

    return [
        total_power,
        power_0_5,
        power_5_15,
        power_15_40,
        peak_freq,
        spec_entropy,
    ]


def get_feature_type_names(include_wavelet: bool = False) -> List[str]:
    feature_names = [
        "mean",
        "std",
        "min",
        "max",
        "range",
        "median",
        "p05",
        "p25",
        "p75",
        "p95",
        "abs_mean",
        "rms",
        "energy",
        "skew",
        "kurtosis",
        "diff_mean",
        "diff_std",
        "diff_abs_max",
        "zero_cross",
        "entropy",
        "total_power",
        "power_0_5",
        "power_5_15",
        "power_15_40",
        "peak_freq",
        "spec_entropy",
    ]
    # wavelet sengaja gak dipakai di script ini
    if include_wavelet:
        raise NotImplementedError("Versi ini fokus no-wavelet biar mapping feature rapi.")
    return feature_names


def extract_features(signal: np.ndarray, fs: float, include_wavelet: bool = False) -> np.ndarray:
    feats: List[float] = []
    for i in range(signal.shape[1]):
        x = signal[:, i]
        feats.extend(base_stats_1d(x))
        feats.extend(spectral_features_1d(x, fs))
    return np.asarray(feats, dtype=np.float32)


def build_feature_table(
    df: pd.DataFrame,
    cfg: Config,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, List[str], List[str], List[Tuple[Any, str]]]:
    X, y, ids = [], [], []
    failed = []
    lead_names_reference = None

    selected_lead_names = None
    feature_type_names = get_feature_type_names(include_wavelet=cfg.include_wavelet_features)

    for _, row in df.iterrows():
        pid = row["patient_id"]
        label = int(row["label"])

        try:
            signal, lead_names, fs = load_ecg_record(pid, cfg.files_dir)

            if cfg.mode == "v1_v2_v3_only":
                idx = lead_indices_from_names(lead_names, ["V1", "V2", "V3"])
                if len(idx) != 3:
                    raise ValueError(f"V1-V3 not found for patient {pid}")
                signal = signal[:, idx]
                selected_lead_names = ["V1", "V2", "V3"]
            else:
                selected_lead_names = lead_names

            if lead_names_reference is None:
                lead_names_reference = selected_lead_names

            feats = extract_features(signal, fs, include_wavelet=cfg.include_wavelet_features)
            X.append(feats)
            y.append(label)
            ids.append(pid)

        except Exception as e:
            failed.append((pid, str(e)))

    if not X:
        raise RuntimeError("No ECG sample loaded successfully.")

    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.int64)
    ids = np.asarray(ids)

    feature_names = []
    for lead in lead_names_reference:
        for feat in feature_type_names:
            feature_names.append(f"{lead}__{feat}")

    return X, y, ids, lead_names_reference, feature_names, failed


# =========================================================
# MODEL
# =========================================================
def build_best_xgb_pipeline(cfg: Config) -> Pipeline:
    # param terbaik dari eksperimen lu sebelumnya
    if cfg.mode == "all_12_leads":
        select_k = 90
        xgb = XGBClassifier(
            random_state=cfg.random_state,
            eval_metric="logloss",
            tree_method="hist",
            n_estimators=250,
            learning_rate=0.03,
            max_depth=4,
            min_child_weight=3,
            subsample=0.7,
            colsample_bytree=1.0,
            reg_alpha=0.0,
            reg_lambda=1.0,
            scale_pos_weight=3.0,
            n_jobs=max(1, os.cpu_count() // 2),
        )
    elif cfg.mode == "v1_v2_v3_only":
        # param terbaik V1-V3 dari classical lu
        select_k = 60
        xgb = XGBClassifier(
            random_state=cfg.random_state,
            eval_metric="logloss",
            tree_method="hist",
            n_estimators=150,
            learning_rate=0.03,
            max_depth=2,
            min_child_weight=3,
            subsample=0.85,
            colsample_bytree=0.6,
            reg_alpha=0.1,
            reg_lambda=2.0,
            scale_pos_weight=3.0,
            n_jobs=max(1, os.cpu_count() // 2),
        )
    else:
        raise ValueError(f"Unknown mode: {cfg.mode}")

    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("select", SelectKBest(score_func=f_classif, k=select_k)),
        ("clf", xgb),
    ])
    return pipe


def evaluate_model(pipe: Pipeline, X_test: np.ndarray, y_test: np.ndarray) -> Dict[str, float]:
    prob = pipe.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.5).astype(int)

    return {
        "roc_auc": float(roc_auc_score(y_test, prob)),
        "pr_auc": float(average_precision_score(y_test, prob)),
        "f1_at_0_5": float(f1_score(y_test, pred, zero_division=0)),
        "balanced_accuracy_at_0_5": float(balanced_accuracy_score(y_test, pred)),
        "precision_at_0_5": float(precision_score(y_test, pred, zero_division=0)),
        "recall_at_0_5": float(recall_score(y_test, pred, zero_division=0)),
    }


# =========================================================
# IMPORTANCE HELPERS
# =========================================================
def get_selected_feature_names(pipe: Pipeline, all_feature_names: List[str]) -> List[str]:
    selector = pipe.named_steps["select"]
    mask = selector.get_support()
    selected = [fname for fname, keep in zip(all_feature_names, mask) if keep]
    return selected


def aggregate_importance_by_lead(feature_names: List[str], importances: np.ndarray) -> pd.DataFrame:
    rows = []
    for fname, imp in zip(feature_names, importances):
        lead, feat = fname.split("__", 1)
        rows.append({
            "lead": lead,
            "feature_type": feat,
            "importance": float(imp),
        })

    df = pd.DataFrame(rows)

    lead_df = (
        df.groupby("lead", as_index=False)["importance"]
        .sum()
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )

    feat_df = (
        df.groupby("feature_type", as_index=False)["importance"]
        .sum()
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )

    return df, lead_df, feat_df


def plot_bar(df: pd.DataFrame, x_col: str, y_col: str, title: str, save_path: Path, top_n: int = None):
    plot_df = df.copy()
    if top_n is not None:
        plot_df = plot_df.head(top_n)

    plt.figure(figsize=(10, 6))
    plt.barh(plot_df[x_col][::-1], plot_df[y_col][::-1])
    plt.title(title)
    plt.xlabel(y_col)
    plt.ylabel(x_col)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()


# =========================================================
# MAIN
# =========================================================
def main():
    output_paths = ensure_output_dirs(CFG)
    save_json(asdict(CFG), output_paths["base"] / "config.json")

    print("=" * 80)
    print("LOAD DATA")
    print("=" * 80)
    df = load_metadata(CFG)
    print("Filtered metadata shape:", df.shape)
    print(df["label"].value_counts().sort_index())
    print()

    X, y, ids, lead_names, feature_names, failed = build_feature_table(df, CFG)
    print("X shape:", X.shape)
    print("y shape:", y.shape)
    print("Lead names used:", lead_names)
    print("Total feature names:", len(feature_names))
    print("Failed records:", len(failed))
    print()

    X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
        X, y, ids,
        test_size=CFG.test_size,
        stratify=y,
        random_state=CFG.random_state,
    )

    print("=" * 80)
    print("TRAIN MODEL")
    print("=" * 80)
    pipe = build_best_xgb_pipeline(CFG)
    pipe.fit(X_train, y_train)

    metrics = evaluate_model(pipe, X_test, y_test)
    print("Holdout metrics:")
    print(json.dumps(metrics, indent=2))
    print()

    # selected feature names after SelectKBest
    selected_feature_names = get_selected_feature_names(pipe, feature_names)
    clf = pipe.named_steps["clf"]

    # 1) model-native importance
    native_importance = clf.feature_importances_
    native_feature_df = pd.DataFrame({
        "feature": selected_feature_names,
        "importance": native_importance,
    }).sort_values("importance", ascending=False).reset_index(drop=True)

    native_full_df, native_lead_df, native_type_df = aggregate_importance_by_lead(
        selected_feature_names, native_importance
    )

    # 2) permutation importance on holdout
    print("=" * 80)
    print("COMPUTE PERMUTATION IMPORTANCE")
    print("=" * 80)
    perm = permutation_importance(
        pipe,
        X_test,
        y_test,
        scoring=CFG.permutation_scoring,
        n_repeats=CFG.permutation_n_repeats,
        random_state=CFG.random_state,
        n_jobs=-1,
    )

    perm_feature_df = pd.DataFrame({
        "feature": feature_names,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std,
    }).sort_values("importance_mean", ascending=False).reset_index(drop=True)

    perm_full_df, perm_lead_df, perm_type_df = aggregate_importance_by_lead(
        feature_names, perm.importances_mean
    )

    # save tables
    native_feature_df.to_csv(output_paths["tables"] / f"native_feature_importance_{CFG.mode}.csv", index=False)
    native_lead_df.to_csv(output_paths["tables"] / f"native_lead_importance_{CFG.mode}.csv", index=False)
    native_type_df.to_csv(output_paths["tables"] / f"native_feature_type_importance_{CFG.mode}.csv", index=False)

    perm_feature_df.to_csv(output_paths["tables"] / f"permutation_feature_importance_{CFG.mode}.csv", index=False)
    perm_lead_df.to_csv(output_paths["tables"] / f"permutation_lead_importance_{CFG.mode}.csv", index=False)
    perm_type_df.to_csv(output_paths["tables"] / f"permutation_feature_type_importance_{CFG.mode}.csv", index=False)

    # plots
    plot_bar(
        native_feature_df,
        x_col="feature",
        y_col="importance",
        title=f"Top {CFG.top_n_features_plot} Native Feature Importance ({CFG.mode})",
        save_path=output_paths["plots"] / f"top_native_feature_importance_{CFG.mode}.png",
        top_n=CFG.top_n_features_plot,
    )

    plot_bar(
        native_lead_df,
        x_col="lead",
        y_col="importance",
        title=f"Native Lead Importance ({CFG.mode})",
        save_path=output_paths["plots"] / f"native_lead_importance_{CFG.mode}.png",
    )

    plot_bar(
        perm_feature_df,
        x_col="feature",
        y_col="importance_mean",
        title=f"Top {CFG.top_n_features_plot} Permutation Feature Importance ({CFG.mode})",
        save_path=output_paths["plots"] / f"top_permutation_feature_importance_{CFG.mode}.png",
        top_n=CFG.top_n_features_plot,
    )

    plot_bar(
        perm_lead_df,
        x_col="lead",
        y_col="importance",
        title=f"Permutation Lead Importance ({CFG.mode})",
        save_path=output_paths["plots"] / f"permutation_lead_importance_{CFG.mode}.png",
    )

    summary = {
        "mode": CFG.mode,
        "metrics": metrics,
        "top_native_features": native_feature_df.head(10).to_dict(orient="records"),
        "top_native_leads": native_lead_df.head(10).to_dict(orient="records"),
        "top_permutation_features": perm_feature_df.head(10).to_dict(orient="records"),
        "top_permutation_leads": perm_lead_df.head(10).to_dict(orient="records"),
    }
    save_json(summary, output_paths["base"] / f"importance_summary_{CFG.mode}.json")

    print("=" * 80)
    print("TOP NATIVE LEAD IMPORTANCE")
    print("=" * 80)
    print(native_lead_df.to_string(index=False))
    print()

    print("=" * 80)
    print("TOP PERMUTATION LEAD IMPORTANCE")
    print("=" * 80)
    print(perm_lead_df.to_string(index=False))
    print()

    print("Saved outputs to:", output_paths["base"].resolve())


if __name__ == "__main__":
    main()

LOAD DATA
Filtered metadata shape: (356, 5)
label
0    287
1     69
Name: count, dtype: int64

X shape: (356, 312)
y shape: (356,)
Lead names used: ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
Total feature names: 312
Failed records: 0

TRAIN MODEL
Holdout metrics:
{
  "roc_auc": 0.7019704433497538,
  "pr_auc": 0.3808504696520169,
  "f1_at_0_5": 0.2608695652173913,
  "balanced_accuracy_at_0_5": 0.5554187192118226,
  "precision_at_0_5": 0.3333333333333333,
  "recall_at_0_5": 0.21428571428571427
}

COMPUTE PERMUTATION IMPORTANCE
TOP NATIVE LEAD IMPORTANCE
lead  importance
  V1    0.125405
  V6    0.122181
  II    0.110208
 aVF    0.109106
  V4    0.097029
 III    0.091031
  V2    0.080210
  V3    0.069880
 aVR    0.066509
  V5    0.063214
   I    0.056777
 aVL    0.008451

TOP PERMUTATION LEAD IMPORTANCE
lead  importance
  V1    0.092426
  V4    0.069335
  V6    0.046429
 III    0.043350
 aVF    0.042241
  V2    0.040702
 aVR    0.025862
  II    0.018596
  